# Fairness and Subgroup Analysis

Checkpoint 44 audits the selected sigmoid-calibrated Logistic Regression across available employee groups. It uses only the 2024 validation period, keeps the 2025 test target locked, and does not assign a binary fair/unfair verdict.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

groups = pd.read_csv(PROCESSED / 'retention_fairness_group_metrics.csv')
disparities = pd.read_csv(PROCESSED / 'retention_fairness_disparities.csv')
comparison = pd.read_csv(PROCESSED / 'retention_fairness_model_comparison.csv')
checks = pd.read_csv(PROCESSED / 'retention_fairness_validation.csv')

selected = groups.loc[groups['model_variant'].eq('Selected model')].copy()
eligible = selected.loc[selected['eligible_for_disparity_comparison'].eq(True)].copy()

## Overall sensitivity comparison

The second model removes age and education only as a diagnostic sensitivity check. It does not replace the selected model.

In [ ]:
comparison[[
    'model_variant',
    'brier_score',
    'pr_auc',
    'roc_auc',
    'top_fraction_precision',
    'top_fraction_capture',
    'captured_positive_cases',
    'selected_for_use',
]].round(4)

## Selected-model subgroup metrics

Selection rate measures how often a group enters the global top 10%. Recall measures how many known attrition cases are captured. The calibration gap is mean predicted probability minus observed attrition.

In [ ]:
selected[[
    'attribute',
    'group',
    'sample_size',
    'positive_cases',
    'selection_rate',
    'true_positive_rate',
    'false_positive_rate',
    'precision',
    'calibration_gap',
    'eligible_for_disparity_comparison',
]].round(4)

## Descriptive disparity summary

These are max-minus-min gaps across groups with sufficient validation evidence. Screening flags request review; they are not legal thresholds or proof of discrimination.

In [ ]:
disparities[[
    'model_variant',
    'attribute',
    'eligible_groups',
    'demographic_parity_difference',
    'equal_opportunity_difference',
    'false_positive_rate_difference',
    'maximum_absolute_calibration_gap',
    'screening_review_flag',
    'review_triggers',
]].round(4)

## Selection-rate visualization

In [ ]:
plotted = eligible.sort_values('selection_rate')
figure, axis = plt.subplots(figsize=(10, 10))
axis.barh(plotted['group_label'], plotted['selection_rate'])
axis.axvline(0.10, color='tab:orange', linestyle='--', label='Overall top-10% rate')
axis.set(xlabel='Selection rate', title='Global Top-10% Selection Rate by Eligible Subgroup')
axis.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axis.legend()
axis.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## Recall visualization

In [ ]:
plotted = eligible.sort_values('true_positive_rate')
overall_capture = float(comparison.loc[comparison['selected_for_use'].eq(True), 'top_fraction_capture'].iloc[0])
figure, axis = plt.subplots(figsize=(10, 10))
axis.barh(plotted['group_label'], plotted['true_positive_rate'])
axis.axvline(overall_capture, color='tab:orange', linestyle='--', label='Overall capture rate')
axis.set(xlabel='Recall', title='Attrition-Case Capture Rate by Eligible Subgroup')
axis.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axis.legend()
axis.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## Calibration visualization

In [ ]:
plotted = eligible.sort_values('calibration_gap')
colors = plotted['calibration_gap'].ge(0).map({True: 'tab:orange', False: 'tab:blue'})
figure, axis = plt.subplots(figsize=(10, 10))
axis.barh(plotted['group_label'], plotted['calibration_gap'], color=colors)
axis.axvline(0, color='black', linewidth=1)
axis.set(
    xlabel='Mean predicted probability minus observed attrition rate',
    title='Subgroup Calibration Gaps',
)
axis.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axis.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## Validation and interpretation

All checks must pass. Large gaps identify areas for investigation, not a fair/unfair verdict. The records are synthetic, missing protected attributes cannot be evaluated, and removing direct fields cannot remove all proxy information.

In [ ]:
checks[['check', 'status', 'observed', 'requirement']]